# Codebase Onboarding Agent —— 代码库入门问答智能体

> 新人接手一个陌生代码库时，能不能直接问「这个功能在哪实现的」「这条调用链怎么走」，并且**每条回答都给出能点回源码的引用**？

这个 notebook 是我一个课程项目在 HelloAgents 框架上的最小可跑复现。原项目是一套 FastAPI + RabbitMQ + PostgreSQL/pgvector 的服务，这里把它压缩成一个单文件流程，但**四条主线的逻辑与原项目一致**：

| Part | 主线 | 原项目里对应的东西 |
|---|---|---|
| 1 | 按语法边界做结构化索引，保留 `文件 / 符号 / 行号` 坐标 | worker 的 AST chunker |
| 2 | BM25 + 向量双路召回，加权 RRF 融合 | 应用层 BM25 + pgvector HNSW |
| 3 | Agent 挂工具做多步取证，引用逐条回索引核验 | LangGraph 5 节点 + 11 个工具 + 引用守卫 |
| 4 | **baseline 阶梯 + 预注册判据的评测** | B0–B4 五级阶梯 + B3.5 诊断臂 |

**Part 4 是这个项目和其他共创项目最不一样的地方。** 它不是"跑几个成功案例给你看"，而是一套能推翻自己的评测：跑之前先锁死什么算通过，跑完老老实实报结论——哪怕结论是"没通过"。

方法论的完整版写在 [Extra14《垂直场景 Agent 的自建评测》](../../Extra-Chapter/Extra14-垂直场景Agent的自建评测.md)，这个 notebook 是它的可执行版本。

## 运行须知

- **Part 1 / 2 / 4 不需要 API Key**，全程本地跑，完全确定性可复现。
- **Part 3（Agent 问答演示）需要 LLM**，走 `HelloAgentsLLM()`，读 `.env`。没配也不影响后面的评测。
- 首次运行会下载约 350MB：语料 tarball + 一个面向代码检索的 embedding 模型。
- 全流程在普通笔记本 CPU 上约 3–5 分钟，Part 4 完全确定性，跑几次结果一样。

---

## 0. 环境准备

In [ ]:
# 首次运行取消注释
# !pip install -q -r requirements.txt

In [ ]:
import ast, io, json, os, re, math, hashlib, tarfile, urllib.request, warnings
from collections import defaultdict, Counter
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

warnings.filterwarnings("ignore")

BASE = Path(".").resolve()
DATA = BASE / "data"
CORPUS_DIR = BASE / "corpus"
print("工作目录:", BASE)

---

# Part 1 · 结构化索引

## 1.1 先把语料钉死

评测的第一条纪律是**冻结被测对象**。语料用 `psf/requests` 的一个具体 commit，不是 `main`——否则今天和下周跑出来的数字没法比。

In [ ]:
CORPUS_REPO = "psf/requests"
CORPUS_COMMIT = "414f0513c33883adf6f2b46901d4f0b38a455851"   # 固定 commit，不用 main

def fetch_corpus() -> Path:
    src_dir = CORPUS_DIR / f"requests-{CORPUS_COMMIT}" / "src" / "requests"
    if src_dir.exists():
        return src_dir
    CORPUS_DIR.mkdir(exist_ok=True)
    url = f"https://codeload.github.com/{CORPUS_REPO}/tar.gz/{CORPUS_COMMIT}"
    print(f"下载语料 {CORPUS_REPO}@{CORPUS_COMMIT[:8]} ...")
    with urllib.request.urlopen(url) as r:
        buf = io.BytesIO(r.read())
    with tarfile.open(fileobj=buf, mode="r:gz") as tar:
        tar.extractall(CORPUS_DIR)
    return src_dir

CORPUS_SRC = fetch_corpus()
py_files = sorted(p for p in CORPUS_SRC.rglob("*.py"))
print(f"语料就绪: {CORPUS_SRC.relative_to(BASE)}")
print(f"Python 文件 {len(py_files)} 个")

## 1.2 按语法边界切块，而不是按固定字符数

固定长度切块会把一个函数从中间劈开，检索命中了也没法给出"这段逻辑在哪个符号里"。

本项目按 Python AST 的结构边界切：模块 docstring、顶层函数、类、类里的方法。**每个 chunk 都带一组稳定坐标 `file_path + symbol_name + start_line`**，后面 BM25、向量检索、引用定位和评测判分全都用这一份坐标，不会各说各话。

注意 `class` chunk 和它的 `method` chunk 是**有意重叠**的：问"这个类负责什么"要看整个类，问"这段逻辑怎么写的"要看具体方法。

In [ ]:
@dataclass
class CodeChunk:
    chunk_id: str
    file_path: str          # 相对语料根，如 "auth.py"
    chunk_type: str         # module_doc | class | function | method
    symbol_name: str
    parent_symbol: Optional[str]
    signature: str
    start_line: int
    end_line: int
    imports: list = field(default_factory=list)
    content: str = ""

    @property
    def label(self) -> str:
        if self.parent_symbol:
            return f"{self.file_path}::{self.parent_symbol}.{self.symbol_name}"
        return f"{self.file_path}::{self.symbol_name}"


def _signature(node) -> str:
    try:
        args = ast.unparse(node.args)
    except Exception:
        args = "..."
    prefix = "async def " if isinstance(node, ast.AsyncFunctionDef) else "def "
    return f"{prefix}{node.name}({args})"


def _segment(lines, node) -> str:
    end = getattr(node, "end_lineno", node.lineno)
    return "\n".join(lines[node.lineno - 1 : end])


def chunk_file(path: Path, rel: str) -> list:
    src = path.read_text(encoding="utf-8", errors="replace")
    try:
        tree = ast.parse(src)
    except SyntaxError:
        # 一个坏文件不该拖垮整个仓库的索引
        print(f"  ⚠️ 跳过（语法错误）: {rel}")
        return []

    lines = src.splitlines()
    imports = []
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            imports += [a.name for a in n.names]
        elif isinstance(n, ast.ImportFrom) and n.module:
            imports.append(n.module)
    imports = sorted(set(imports))

    out, seq = [], 0

    def emit(**kw):
        nonlocal seq
        seq += 1
        out.append(CodeChunk(chunk_id=f"{rel}#{seq}", file_path=rel,
                             imports=imports, **kw))

    doc = ast.get_docstring(tree)
    if doc:
        emit(chunk_type="module_doc", symbol_name=Path(rel).stem,
             parent_symbol=None, signature=f"module {rel}",
             start_line=1, end_line=min(len(lines), 20), content=doc)

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            bases = ", ".join(getattr(b, "id", "?") for b in node.bases)
            emit(chunk_type="class", symbol_name=node.name, parent_symbol=None,
                 signature=f"class {node.name}({bases})",
                 start_line=node.lineno, end_line=node.end_lineno,
                 content=_segment(lines, node))
            for sub in node.body:
                if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    emit(chunk_type="method", symbol_name=sub.name,
                         parent_symbol=node.name, signature=_signature(sub),
                         start_line=sub.lineno, end_line=sub.end_lineno,
                         content=_segment(lines, sub))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            emit(chunk_type="function", symbol_name=node.name, parent_symbol=None,
                 signature=_signature(node),
                 start_line=node.lineno, end_line=node.end_lineno,
                 content=_segment(lines, node))
    return out


CHUNKS = []
for p in py_files:
    CHUNKS += chunk_file(p, str(p.relative_to(CORPUS_SRC)))

BY_ID = {c.chunk_id: c for c in CHUNKS}
print(f"共 {len(CHUNKS)} 个 chunk")
print("类型分布:", dict(Counter(c.chunk_type for c in CHUNKS)))

In [ ]:
# 看一个具体的 chunk：坐标齐全，能直接点回源码
demo = next(c for c in CHUNKS if c.symbol_name == "prepare_auth")
print(f"label     : {demo.label}")
print(f"type      : {demo.chunk_type}")
print(f"signature : {demo.signature}")
print(f"lines     : {demo.start_line}–{demo.end_line}")
print("-" * 60)
print("\n".join(demo.content.splitlines()[:12]))

---

# Part 2 · 混合检索

## 2.1 为什么要两路

代码库的问题天然有两类：

- 用户写得出符号名（`prepare_auth`、`HTTPAdapter`、某个配置键）→ **精确词面匹配**更有效
- 用户只描述行为（"重定向时怎么处理 Authorization 头"）→ **语义召回**更有效

单独一路都会漏。所以保留 BM25 和向量两条**可以各自单独评测**的检索臂，再融合。

In [ ]:
from rank_bm25 import BM25Okapi

_CAMEL = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")

def tokenize(text: str) -> list:
    '''代码检索的分词：先按非字母数字切，再拆 camelCase，最后小写。
    `prepareAuth` / `prepare_auth` / `PrepareAuth` 会落到同一批 token 上。'''
    parts = re.split(r"[^A-Za-z0-9]+", text)
    out = []
    for p in parts:
        if not p:
            continue
        out += [s.lower() for s in _CAMEL.split(p) if s]
    return out


def bm25_text(c: CodeChunk) -> str:
    return f"{c.file_path} {c.parent_symbol or ''} {c.symbol_name} {c.signature}\n{c.content}"

BM25_CORPUS = [tokenize(bm25_text(c)) for c in CHUNKS]
BM25 = BM25Okapi(BM25_CORPUS)
print(f"BM25 索引就绪，{len(BM25_CORPUS)} 篇文档")

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 原项目用的是 jinaai/jina-embeddings-v2-base-code（768 维，代码专用）。
# notebook 换成同样面向代码检索、但更小的 CodeSearchNet 模型，CPU 上一分钟内能跑完。
# 别用通用文本模型：实测换成 all-MiniLM-L6-v2 后 B3 的 composite 从 0.742 掉到 0.593。
EMBED_MODEL = "flax-sentence-embeddings/st-codesearch-distilroberta-base"

def embed_text(c: CodeChunk) -> str:
    head = f"{c.file_path} {c.signature}"
    return f"{head}\n{c.content[:1500]}"

print(f"加载 embedding 模型 {EMBED_MODEL} ...")
encoder = SentenceTransformer(EMBED_MODEL)
DENSE = encoder.encode([embed_text(c) for c in CHUNKS],
                       batch_size=64, normalize_embeddings=True,
                       show_progress_bar=True)
print("向量矩阵:", DENSE.shape)   # 内存版的 pgvector

## 2.2 加权 RRF：为什么不直接把两个分数加起来

BM25 的分数和余弦相似度**尺度和分布都不一样**，直接相加或者归一化后相加都不稳：min-max 会被当前查询的极值带跑偏，z-score 对重尾的 BM25 分数不可靠。

RRF 只用**名次**，不用原始分数，所以不需要假设两边同分布。代价是丢掉了分数幅度信息。

两个参数说明：`K=60` 是常用默认值；`dense:sparse = 2:1` 是实测调出来的——等权时稀疏侧噪声太多，纯稠密臂反而在 nDCG 上反超混合臂。**这两个数没有做过系统性超参搜索**，换语料要重新验。

In [ ]:
RRF_K = 60
W_DENSE, W_SPARSE = 2.0, 1.0

def bm25_rank(query: str, n: int = 50) -> list:
    scores = BM25.get_scores(tokenize(query))
    idx = np.argsort(scores)[::-1][:n]
    return [CHUNKS[i].chunk_id for i in idx if scores[i] > 0]

def dense_rank(query: str, n: int = 50) -> list:
    qv = encoder.encode([query], normalize_embeddings=True)[0]
    scores = DENSE @ qv
    idx = np.argsort(scores)[::-1][:n]
    return [CHUNKS[i].chunk_id for i in idx]

def weighted_rrf(rank_lists, weights, k=RRF_K) -> list:
    acc = defaultdict(float)
    for ranked, w in zip(rank_lists, weights):
        for rank, cid in enumerate(ranked, start=1):
            acc[cid] += w / (k + rank)
    return [cid for cid, _ in sorted(acc.items(), key=lambda kv: -kv[1])]

def hybrid_rank(query: str, n: int = 50) -> list:
    return weighted_rrf([dense_rank(query, n), bm25_rank(query, n)],
                        [W_DENSE, W_SPARSE])

In [ ]:
q = "How is basic auth attached to a prepared request?"
print("BM25 前 5:")
for cid in bm25_rank(q)[:5]:
    print("   ", BY_ID[cid].label)
print("\nDense 前 5:")
for cid in dense_rank(q)[:5]:
    print("   ", BY_ID[cid].label)
print("\n加权 RRF 前 5:")
for cid in hybrid_rank(q)[:5]:
    print("   ", BY_ID[cid].label)

---

# Part 3 · Agent、工具与引用验证

一次 top-k 检索能回答"这个函数在哪"，但回答不了"这条链路怎么走"——那需要**接着往下查**：看到 `prepare_auth` 调用了 `auth(self)`，就得去找 `HTTPBasicAuth.__call__`，再找到 `_basic_auth_str`。

所以给 Agent 四个只读工具，让它自己决定查几步。

In [ ]:
# ---- 静态调用点：从每个 chunk 里抽出它调用了哪些名字（程序分析，不是模型推断）
CALL_SITES = defaultdict(set)     # symbol -> {chunk_id 调用了它}
DEFINED_BY = defaultdict(list)    # symbol -> [chunk_id 定义了它]

for c in CHUNKS:
    if c.chunk_type in ("class", "function", "method"):
        DEFINED_BY[c.symbol_name].append(c.chunk_id)
    try:
        for node in ast.walk(ast.parse(c.content)):
            if isinstance(node, ast.Call):
                f = node.func
                name = getattr(f, "id", None) or getattr(f, "attr", None)
                if name:
                    CALL_SITES[name].add(c.chunk_id)
    except SyntaxError:
        pass

resolved = sum(1 for s in CALL_SITES if s in DEFINED_BY)
print(f"调用点涉及 {len(CALL_SITES)} 个名字，其中 {resolved} 个能在本仓库内解析到定义")
print(f"解析率 {resolved / len(CALL_SITES):.1%}  ← 纯静态 AST 没有类型推断，动态分派解析不了，策略是宁可少报")

In [ ]:
def tool_search_code(query: str) -> str:
    '''按自然语言或符号名检索代码，返回带 evidence ID 的候选。'''
    out = []
    for cid in hybrid_rank(query)[:5]:
        c = BY_ID[cid]
        out.append(f"[{cid}] {c.label}  (L{c.start_line}-{c.end_line})\n{c.signature}")
    return "\n\n".join(out) if out else "no match"

def tool_file_outline(file_path: str) -> str:
    '''列出某个文件里的所有符号，用来快速建立文件结构印象。'''
    items = [c for c in CHUNKS if c.file_path == file_path]
    if not items:
        return f"file not found: {file_path}"
    return "\n".join(f"[{c.chunk_id}] L{c.start_line:>4} {c.chunk_type:<9} {c.label}"
                     for c in sorted(items, key=lambda x: x.start_line))

def tool_read_symbol(symbol: str) -> str:
    '''读一个符号的完整源码。'''
    ids = DEFINED_BY.get(symbol, [])
    if not ids:
        return f"symbol not found: {symbol}"
    c = BY_ID[ids[0]]
    body = "\n".join(c.content.splitlines()[:60])
    return f"[{c.chunk_id}] {c.label} (L{c.start_line}-{c.end_line})\n{body}"

def tool_find_callers(symbol: str) -> str:
    '''查哪些地方调用了这个符号——跨文件追链路主要靠它。'''
    ids = sorted(CALL_SITES.get(symbol, []))[:8]
    if not ids:
        return f"no caller found for: {symbol}"
    return "\n".join(f"[{i}] {BY_ID[i].label}" for i in ids)

print(tool_find_callers("_basic_auth_str"))

## 3.1 引用验证：两个必须堵死的口子

Agent 给出答案后，每条引用都要回索引核验——evidence ID 存不存在、指向的符号对不对。验不过的引用从正文删掉。

这里有两个反直觉但很关键的规则，**它们保护的是指标的诚实性，不是用户体验**：

1. **计分用删除前的引用集合。** 如果按删除后计分，系统就能先大量生成引用、错的反正会被删掉，剩下的正确率自然高——用"多猜"换高分。
2. **零引用答案记 0 分，而不是满分。** groundedness 衡量"每句话是否有据"，一个什么都不引用的答案技术上没有任何无据断言，旧逻辑会给它满分——于是"不回答"成了最优策略。

这两条在原项目里都是踩过坑才加上的，细节见 Extra14 第七章。

In [ ]:
CITE_RE = re.compile(r"\[([A-Za-z0-9_./\\-]+#\d+)\]")

def verify_citations(answer: str):
    raw = CITE_RE.findall(answer)                      # 删除前
    verified = [cid for cid in raw if cid in BY_ID]     # 删除后
    cleaned = answer
    for cid in raw:
        if cid not in BY_ID:
            cleaned = cleaned.replace(f"[{cid}]", "")
    groundedness = 0.0 if not raw else len(verified) / len(raw)
    if not raw:
        groundedness = 0.0        # 规则 2：零引用不给满分
    return {
        "cited_before_filter": raw,      # 规则 1：计分用这个
        "verified": verified,
        "answer": cleaned,
        "groundedness": groundedness,
    }

demo_answer = ("Basic auth is applied in [models.py#33] via the auth callable, "
               "which builds the header in [auth.py#2], and [nonexistent.py#9] is fake.")
r = verify_citations(demo_answer)
print("删除前引用:", r["cited_before_filter"])
print("验证通过  :", r["verified"])
print("groundedness:", round(r["groundedness"], 3))
print("清洗后答案:", r["answer"])

## 3.2 接上 HelloAgents 的 Agent

下面这段需要 LLM。没配 `.env` 会跳过，不影响 Part 4 的评测。

In [ ]:
SYSTEM_PROMPT = '''You are a code onboarding assistant for the `requests` library.

Rules:
1. Answer ONLY from evidence returned by tools. Never invent file names or line numbers.
2. Every factual claim must carry an evidence ID in square brackets, e.g. [auth.py#2].
3. Use search_code first, then follow the chain with find_callers / read_symbol.
4. If the evidence is insufficient, say so explicitly instead of guessing.'''

agent = None
try:
    from dotenv import load_dotenv
    load_dotenv()
    from hello_agents import ReActAgent, HelloAgentsLLM, ToolRegistry

    registry = ToolRegistry()
    registry.register_function("search_code",
        "Search the codebase by natural language or symbol name. Returns candidates with evidence IDs.",
        tool_search_code)
    registry.register_function("file_outline",
        "List all symbols in a file, e.g. 'sessions.py'.", tool_file_outline)
    registry.register_function("read_symbol",
        "Read the full source of a symbol by name, e.g. 'prepare_auth'.", tool_read_symbol)
    registry.register_function("find_callers",
        "Find which code locations call a given symbol.", tool_find_callers)

    agent = ReActAgent(
        name="Codebase Onboarding Agent",
        llm=HelloAgentsLLM(),
        system_prompt=SYSTEM_PROMPT,
        tool_registry=registry,
        max_steps=14,          # 硬预算，失败调用也计步
    )
    print("✅ Agent 就绪，4 个工具已注册")
except Exception as e:
    print(f"⚠️ 未启用 Agent（{type(e).__name__}: {e}）")
    print("   Part 4 的评测不依赖 LLM，可以直接往下跑。")

In [ ]:
if agent is not None:
    question = "When following a redirect to a different host, how does a Session decide whether to strip the Authorization header?"
    raw_answer = agent.run(question)

    checked = verify_citations(str(raw_answer))
    print("\n" + "=" * 70)
    print(checked["answer"])
    print("=" * 70)
    print(f"引用（删除前）: {len(checked['cited_before_filter'])} 条")
    print(f"验证通过      : {len(checked['verified'])} 条")
    print(f"groundedness  : {checked['groundedness']:.2f}")
    for cid in checked["verified"]:
        c = BY_ID[cid]
        print(f"  → {c.label}  L{c.start_line}-{c.end_line}")
else:
    print("跳过（未配置 LLM）")

---

# Part 4 · 评测：这个系统真的比普通 RAG 好吗

到这儿为止，我可以挑三个回答得漂亮的问题截图，配一句"系统能够准确理解代码库语义"，然后收工。

那只能证明它偶尔能工作。回答不了三个更要紧的问题：

1. 它比一次普通的向量 RAG 好多少？
2. 如果好，好在检索融合，还是多步取证？
3. 换一天再跑，结论还成立吗？

下面这套流程就是为了回答它们。

## 4.1 一道题长什么样

不是一段"标准答案文字"——那得靠 LLM Judge 去比对，又贵又抖。

用**锚点**：每道题给出若干个正确的源码位置 `file + symbol + start_line`。系统最终引用命中了几个 → `Recall@5`；第一条正确证据排多靠前 → `MRR`；多条正确证据的整体排序 → `nDCG@5`。全部可自动判分，零 LLM 成本。

代价说在前面：**这套题测的是"有没有找到并引用正确代码"，不测"解释写得好不好"。** 范围要讲清楚，不能宣称测了没测的东西。

In [ ]:
QUESTIONS_PATH = DATA / "questions.jsonl"
QUESTIONS = [json.loads(l) for l in QUESTIONS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

# 题集 SHA：跑完对一次，确保中途没人（包括我自己）偷偷改题
QUESTIONS_SHA256 = hashlib.sha256(QUESTIONS_PATH.read_bytes()).hexdigest()

print(f"题集 {len(QUESTIONS)} 道，SHA256 {QUESTIONS_SHA256[:16]}…")
print("层级分布:", dict(Counter(q["taxonomy"] for q in QUESTIONS)))
print("来源分布:", dict(Counter(q["source"] for q in QUESTIONS)))
print()
sample = QUESTIONS[2]
print("示例题:", sample["question"])
for t in sample["gt_targets"]:
    print(f"   gold → {t['file']}::{t['symbol']}  L{t['start_line']}")

> **题集偏置要主动说。** 10 道里有 4 道标了 `graph_reverse`——从系统已经建出的调用关系反推出来的问题。这类题天然不会包含**系统本身漏建的链路**，而且它们全落在 L2/L3，来源和难度是混淆的。它能高效增加图敏感的问题，但不能证明题集无偏。
>
> 一份不写偏置的评测报告，不值得信。

In [ ]:
def anchor_match(c: CodeChunk, gold: dict) -> bool:
    '''锚点匹配。这个函数决定了所有指标——放松一点，所有分数一起变好。
    所以它应该被单独 review、单独测试，不要藏在评估器里随手改。'''
    if c.file_path != gold["file"]:
        return False
    if c.symbol_name == gold["symbol"]:
        return True
    # 类 chunk 覆盖其方法，会命中方法锚点。这是有意的重叠，但它让匹配偏松，需在报告里说明。
    return c.start_line <= gold["start_line"] <= c.end_line


def recall_at_k(ranked_ids, golds, k=5) -> float:
    top = [BY_ID[i] for i in ranked_ids[:k]]
    hit = sum(1 for g in golds if any(anchor_match(c, g) for c in top))
    return hit / len(golds) if golds else 0.0

def mrr(ranked_ids, golds, k=5) -> float:
    for rank, cid in enumerate(ranked_ids[:k], start=1):
        if any(anchor_match(BY_ID[cid], g) for g in golds):
            return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked_ids, golds, k=5) -> float:
    # 每个 gold 只能被记一次分。否则一个 gold 被多个 chunk 命中时（类 chunk 和
    # 它的方法 chunk 本来就重叠），DCG 会超过 ideal，nDCG 冒出大于 1 的值。
    # 写这个 notebook 时真的踩到了：有一题的 composite 一度是 1.038。
    covered, gains = set(), []
    for cid in ranked_ids[:k]:
        c = BY_ID[cid]
        hit = next((i for i, g in enumerate(golds)
                    if i not in covered and anchor_match(c, g)), None)
        if hit is None:
            gains.append(0.0)
        else:
            covered.add(hit)
            gains.append(1.0)
    dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1.0 / math.log2(i + 2) for i in range(min(len(golds), k)))
    return dcg / ideal if ideal else 0.0

## 4.2 baseline 阶梯

跟谁比？

不跟 Claude Code、Cursor 比。它们的模型、上下文、私有检索和权限我都统一不了，赢了不知道赢在哪，输了也不知道输在哪——那是产品赛马，不是评测。

要比就比**控制变量后的参考系统**，一级一级加能力：

| 臂 | 路径 | 作用 |
|---|---|---|
| **B2** | dense-only 检索 | 普通向量 RAG，主要对手 |
| **B3** | BM25 + dense + 加权 RRF | 强检索 baseline，最难打的对手 |
| **B4** | B3 + 多步取证（符号跳转 + 调用点展开） | 完整系统 |

**关键不在任何一个臂，而在这句话：三个臂共用同一份索引、同一套 chunk 坐标、同一个判分函数，差异只在检索路径。** 只有这样，`B4 − B3` 才有解释力。

原项目里的阶梯是 B0–B4 五级外加一个 B3.5 诊断臂，四个 Agent 臂还共用同一个合成模型与 prompt。这里的 B4 用**确定性规划器**——不是偷懒，是原项目也这么做的：LLM 选工具会额外引入工具选择方差，做消融时会把系统的贡献和模型的抖动混在一起。

In [ ]:
TOP_K = 5

def arm_B2(question: str) -> list:
    '''B2：dense-only，普通向量 RAG。'''
    return dense_rank(question, 50)[:TOP_K]

def arm_B3(question: str) -> list:
    '''B3：BM25 + dense + 加权 RRF。'''
    return hybrid_rank(question, 50)[:TOP_K]

POS = {c.chunk_id: i for i, c in enumerate(CHUNKS)}

def arm_B4(question: str, budget: int = 14) -> list:
    '''B4：在 B3 之上做确定性多步取证。

    第 1 步 混合检索拿种子候选
    第 2 步 从前 6 名种子里抽出它调用的名字，把这些符号的**定义**追进来
    第 3 步 问题里直接写出的符号名做一次跳转，来源标记为最强
    最后    追出来的证据按「来源名次 + 自身相关度」排成一个列表，再与种子做 RRF

    每一步计入 14 步预算，失败也计步。

    有个细节值得单独说：追出来的这一堆证据**不是一个有序列表**，
    如果直接按 ast.walk 的遍历顺序丢进 RRF，名次就成了噪声，好证据会被挤下去。
    所以要先按「它是从第几名种子追出来的」排序，让证据链的强弱体现在名次上。
    我第一版就是错的，B4 因此一直低于 B3。

    规划器是确定性的——原项目也这么做，因为让 LLM 选工具会引入工具选择方差，
    做消融时会把系统的贡献和模型的抖动混在一起。
    '''
    steps = 0
    seed = hybrid_rank(question, 50)
    steps += 1

    provenance = {}                                   # chunk_id -> 最好的来源名次
    for rank, cid in enumerate(seed[:6], start=1):    # 沿调用关系向外走一层
        if steps >= budget:
            break
        steps += 1
        try:
            tree = ast.parse(BY_ID[cid].content)
        except SyntaxError:
            continue
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                name = getattr(node.func, "id", None) or getattr(node.func, "attr", None)
                for target in DEFINED_BY.get(name, [])[:1]:
                    provenance.setdefault(target, rank)

    for token in sorted(set(re.findall(r"[A-Za-z_][A-Za-z0-9_]{3,}", question))):
        if steps >= budget:
            break
        if token in DEFINED_BY:                       # 问题里直接点名的符号，来源最强
            steps += 1
            for target in DEFINED_BY[token][:1]:
                provenance[target] = 0

    qv = encoder.encode([question], normalize_embeddings=True)[0]
    followed = sorted(provenance,
                      key=lambda cid: (provenance[cid], -float(DENSE[POS[cid]] @ qv)))
    return weighted_rrf([seed, followed], [3.0, 1.0])[:TOP_K]

ARMS = {"B2": arm_B2, "B3": arm_B3, "B4": arm_B4}
print("进判据的对照臂：", list(ARMS))

### 加一个诊断臂：查询改写值多少

Jerry 式的追问：BM25 加向量之外，查询改写呢？

自然的做法是把改写当 feature 加进 B3 然后说"我做了查询改写"。**更有用的做法是把它做成一个臂，测出它值多少。**

这里的改写是确定性的三步，不调 LLM——理由和确定性规划器一样：让 LLM 改写会引入额外方差，做消融时会把系统贡献和模型抖动混在一起。

1. **同义扩展**：按代码命名习惯，把自然语言动词扩到常见函数名用词（`strip` → `remove / clear / rebuild`）
2. **符号表回填**：问题里的词与真实符号名有两个及以上重合时，把该符号名补进查询
3. **只扩不换**：原问题始终保留

In [ ]:
# 代码检索场景的词表。真实系统会用领域词典或 LLM 生成，这里手写以保证确定性。
CODE_SYNONYMS = {
    "build": ["make", "create", "construct", "prepare"],
    "create": ["make", "new", "init", "prepare"],
    "check": ["verify", "validate", "is", "has"],
    "decide": ["check", "verify", "should", "resolve"],
    "get": ["fetch", "read", "load", "resolve"],
    "set": ["update", "write", "apply"],
    "handle": ["process", "dispatch", "resolve", "rebuild"],
    "strip": ["remove", "delete", "clear", "pop", "rebuild"],
    "extract": ["parse", "read", "get"],
    "merge": ["update", "combine", "apply"],
    "store": ["save", "set", "update", "jar"],
    "follow": ["resolve", "redirect", "next"],
    "infer": ["get", "detect", "guess", "from"],
    "trace": ["call", "send", "request", "dispatch"],
    "path": ["flow", "chain", "send"],
    "encoding": ["charset", "decode", "encode"],
    "header": ["headers"],
    "redirect": ["redirects", "location", "resolve"],
    "auth": ["authorization", "authenticate", "credentials", "basic"],
    "authorization": ["auth", "credentials"],
    "cookie": ["cookies", "cookiejar", "jar"],
    "proxy": ["proxies", "bypass", "environment"],
    "cert": ["certificate", "ssl", "tls", "verify"],
    "certificate": ["cert", "ssl", "tls", "verify"],
    "body": ["data", "payload", "content"],
    "session": ["sessions"],
    "adapter": ["adapters", "transport"],
}

MAX_EXPAND = 12

def rewrite_query(question: str, max_expand: int = MAX_EXPAND):
    '''确定性查询改写，返回 (改写后的查询, 新增的词)。'''
    toks = set(tokenize(question))
    extra = []
    for t in sorted(toks):        # 必须排序：set 的迭代顺序随 PYTHONHASHSEED 变，
        extra += CODE_SYNONYMS.get(t, [])   # 不排序的话截断到 max_expand 时选中的词每次都不同
    for sym in DEFINED_BY:                       # 符号表回填
        if len(set(tokenize(sym)) & toks) >= 2:
            extra.append(sym)
    extra = [e for e in dict.fromkeys(extra) if e not in toks][:max_expand]
    return (question + " " + " ".join(extra)).strip(), extra

demo_q = "How is basic auth attached to a prepared request?"
print(demo_q)
print("扩展出：", rewrite_query(demo_q)[1])

改写要喂给哪一路？这是个容易想当然的地方，所以拆成两个臂分别测：

- **B3Q**：改写后的查询同时给稀疏臂和稠密臂
- **B3Qs**：只给稀疏臂，稠密臂仍用原问题

依据是：查询扩展本质是**词面**技术。把一串关键词塞进查询，BM25 会受益，但稠密臂拿到的句向量会偏离原始意图。

两个臂都是**诊断臂**，和第 4.3 节锁定的判据无关——判据只认 B2、B3、B4。这一点和第五章那条纪律一样：后加的臂可以改变我对系统的理解，不能改变已经锁定的判定。

In [ ]:
def arm_B3Q(question: str) -> list:
    '''诊断臂：改写后的查询给两路。'''
    return hybrid_rank(rewrite_query(question)[0], 50)[:TOP_K]

def arm_B3Qs(question: str) -> list:
    '''诊断臂：只给稀疏臂改写，稠密臂用原问题。'''
    rq, _ = rewrite_query(question)
    return weighted_rrf([dense_rank(question, 50), bm25_rank(rq, 50)],
                        [W_DENSE, W_SPARSE])[:TOP_K]

def arm_B4Qs(question: str) -> list:
    '''诊断臂：改写(仅稀疏) + 多步取证，看两个增强叠不叠加。'''
    rq, _ = rewrite_query(question)
    seed = weighted_rrf([dense_rank(question, 50), bm25_rank(rq, 50)], [W_DENSE, W_SPARSE])
    steps, provenance = 1, {}
    for rank, cid in enumerate(seed[:6], start=1):
        if steps >= 14:
            break
        steps += 1
        try:
            tree = ast.parse(BY_ID[cid].content)
        except SyntaxError:
            continue
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                name = getattr(node.func, "id", None) or getattr(node.func, "attr", None)
                for target in DEFINED_BY.get(name, [])[:1]:
                    provenance.setdefault(target, rank)
    for token in sorted(set(re.findall(r"[A-Za-z_][A-Za-z0-9_]{3,}", rq))):
        if steps >= 14:
            break
        if token in DEFINED_BY:
            steps += 1
            for target in DEFINED_BY[token][:1]:
                provenance[target] = 0
    qv = encoder.encode([question], normalize_embeddings=True)[0]
    followed = sorted(provenance, key=lambda cid: (provenance[cid], -float(DENSE[POS[cid]] @ qv)))
    return weighted_rrf([seed, followed], [3.0, 1.0])[:TOP_K]

DIAGNOSTIC = {"B3Q": arm_B3Q, "B3Qs": arm_B3Qs, "B4Qs": arm_B4Qs}
print("诊断臂（不进判据）：", list(DIAGNOSTIC))

## 4.3 预注册判据 —— 这一格必须在跑之前执行

跑完再决定"什么算成功"，你一定会挑一个对自己有利的说法。数据在手上的时候，人的自我说服能力强得可怕。

所以在看到任何结果之前，把成功标准写成一条**可执行的规则**：

In [ ]:
DECISION_RULE = {
    "treatment": "B4",
    "controls": ["B2", "B3"],          # 两个对手都要打过
    "strata": ["L2", "L3"],            # 只看跨文件和架构题；L1 一次检索就够，属于送分
    "min_margin": 0.05,                # 工程门槛，不是统计显著性阈值
    "composite": ["recall@5", "mrr", "ndcg@5"],   # 三项均值
}

# 2 层 × 2 个对手 = 4 项比较，任意一项不满足 → 整体 unsupported
N_COMPARISONS = len(DECISION_RULE["strata"]) * len(DECISION_RULE["controls"])

PROVENANCE = {
    "corpus": f"{CORPUS_REPO}@{CORPUS_COMMIT}",
    "n_files": len(py_files),
    "n_chunks": len(CHUNKS),
    "questions_sha256": QUESTIONS_SHA256,
    "embed_model": EMBED_MODEL,
    "rrf": {"K": RRF_K, "dense": W_DENSE, "sparse": W_SPARSE},
    "top_k": TOP_K,
    "decision_rule": DECISION_RULE,
}
print(json.dumps(PROVENANCE, indent=2, ensure_ascii=False))
print(f"\n共 {N_COMPARISONS} 项比较，全过才算 supported。")

In [ ]:
def run_suite() -> dict:
    per_q = []
    for q in QUESTIONS:
        row = {"id": q["id"], "taxonomy": q["taxonomy"], "source": q["source"]}
        for name, fn in {**ARMS, **DIAGNOSTIC}.items():
            ranked = fn(q["question"])
            row[name] = {
                "recall@5": recall_at_k(ranked, q["gt_targets"], TOP_K),
                "mrr": mrr(ranked, q["gt_targets"], TOP_K),
                "ndcg@5": ndcg_at_k(ranked, q["gt_targets"], TOP_K),
                "top": [BY_ID[c].label for c in ranked],
            }
        per_q.append(row)
    return {"per_question": per_q}

def composite(cell: dict) -> float:
    return sum(cell[m] for m in DECISION_RULE["composite"]) / len(DECISION_RULE["composite"])

RESULT = run_suite()
n_arms = len(ARMS) + len(DIAGNOSTIC)
print(f"完成 {len(QUESTIONS)} 题 × {n_arms} 臂 = {len(QUESTIONS) * n_arms} 次执行")

## 4.4 结果

In [ ]:
import pandas as pd

overall = []
for name in {**ARMS, **DIAGNOSTIC}:
    cells = [r[name] for r in RESULT["per_question"]]
    overall.append({
        "臂": name,
        "Recall@5": round(sum(c["recall@5"] for c in cells) / len(cells), 3),
        "MRR":      round(sum(c["mrr"] for c in cells) / len(cells), 3),
        "nDCG@5":   round(sum(c["ndcg@5"] for c in cells) / len(cells), 3),
        "composite": round(sum(composite(c) for c in cells) / len(cells), 3),
    })
df_overall = pd.DataFrame(overall).set_index("臂")
print("整体阶梯（10 题全体）")
display(df_overall)

In [ ]:
rows = []
for stratum in DECISION_RULE["strata"]:
    subset = [r for r in RESULT["per_question"] if r["taxonomy"] == stratum]
    if not subset:
        continue
    row = {"层级": stratum, "题数": len(subset)}
    for name in {**ARMS, **DIAGNOSTIC}:
        row[name] = round(sum(composite(r[name]) for r in subset) / len(subset), 4)
    for ctrl in DECISION_RULE["controls"]:
        row[f"B4−{ctrl}"] = round(row["B4"] - row[ctrl], 4)
    rows.append(row)

df_strata = pd.DataFrame(rows).set_index("层级")
print("分层 composite 与 margin")
display(df_strata)

In [ ]:
# 按预注册规则判定，不看结果好坏
checks, passed = [], 0
for row in rows:
    for ctrl in DECISION_RULE["controls"]:
        margin = row[f"B4−{ctrl}"]
        ok = margin >= DECISION_RULE["min_margin"]
        passed += ok
        checks.append({
            "层级": row["层级"],
            "比较": f"B4 vs {ctrl}",
            "margin": margin,
            "门槛": DECISION_RULE["min_margin"],
            "通过": "✅" if ok else "❌",
        })

display(pd.DataFrame(checks))

verdict = "supported" if passed == N_COMPARISONS else "unsupported"
print(f"\n四项比较通过 {passed}/{N_COMPARISONS}")
print(f"H1 verdict: {verdict.upper()}")

### 判据之后，再看诊断臂

顺序很重要：**先出预注册判据的结论，再看诊断臂。** 反过来做，就会忍不住用诊断臂的好结果去重新解释判据。

In [ ]:
# 两个增强各自相对 B3 的孤立贡献
iso = []
for row in rows:
    st = row["层级"]
    iso.append({
        "层级": st,
        "改写(两路都改) B3Q−B3": round(row["B3Q"] - row["B3"], 4),
        "改写(仅稀疏) B3Qs−B3": round(row["B3Qs"] - row["B3"], 4),
        "多步取证 B4−B3": round(row["B4"] - row["B3"], 4),
        "两者叠加 B4Qs−B3": round(row["B4Qs"] - row["B3"], 4),
    })
display(pd.DataFrame(iso).set_index("层级"))

In [ ]:
# 扩展词数的敏感性。注意：这是在同一批题上扫的，见下面的说明
sens = []
for budget in (4, 8, 12, 20):
    tot = 0.0
    for q in QUESTIONS:
        rq = rewrite_query(q["question"], budget)[0]
        ranked = weighted_rrf([dense_rank(q["question"], 50), bm25_rank(rq, 50)],
                              [W_DENSE, W_SPARSE])[:TOP_K]
        tot += composite({m: fn(ranked, q["gt_targets"], TOP_K) for m, fn in
                          (("recall@5", recall_at_k), ("mrr", mrr), ("ndcg@5", ndcg_at_k))})
    sens.append({"max_expand": budget, "整体 composite": round(tot / len(QUESTIONS), 4)})
display(pd.DataFrame(sens).set_index("max_expand"))

三个结论，都不太符合直觉：

**一、朴素的查询改写是负收益。** 把扩展词同时喂给两路，整体 composite 从 0.686 掉到 0.667——**比不改还差**。

最能说明问题的是 q03「basic auth 怎么挂到 prepared request 上」：改写扩展出的词里**正好包含 gold 符号 `_basic_auth_str`**，可这一题的分数从 0.790 掉到 0.472。加对了关键词反而更差。

**二、只改稀疏臂就变正了。** B3Qs 整体 0.700，L2 上相对 B3 是 **+0.066**，超过 0.05 门槛；q03 纹丝不动停在 0.790。

原因是机制层面的：查询扩展是**词面**技术。BM25 拿到更多关键词会受益；而稠密臂拿到的是一串堆砌关键词的句子，句向量偏离了原始意图。**"把改写后的查询喂给整条检索链"是个默认选项，但它是错的。**

**三、和多步取证不叠加。** 改写在 L2 上 +0.066、多步取证 −0.042，两个一起上（B4Qs）整体只有 0.657，比两者单独都低；L3 上叠加更是 −0.105。它们抢的是同一批 top-5 名额。

### 关于上面那张敏感性表

`max_expand` 扫下来 8 最好（0.731），比默认的 12（0.700）高。**但我没有把默认值改成 8。**

因为这个扫描是在同一批 10 道题上做的——**用测试集选超参，等于把测试集变成了开发集**，选出来的 0.731 不能当作一个可信的效果。要定这个参数得有独立的开发集。

所以这里保留最初选的 12，把敏感性原样报出来。这也是这一格值得跑一遍的原因：它让"我调了个参数，涨了 0.03"这件事看起来到底有多可疑，变得直观。

> 顺带一个真踩到的坑：`rewrite_query` 里原本写的是 `for t in set(...)`。**Python 的 set 迭代顺序随 `PYTHONHASHSEED` 变**，截断到 `max_expand` 时选中的词每次运行都不一样，这张敏感性表连跑两次能差 0.02。评测代码里凡是"取前 N 个"，前面那个集合必须是有序的。

In [ ]:
# 逐题看，才知道均值掩盖了什么
detail = []
for r in RESULT["per_question"]:
    detail.append({
        "id": r["id"], "层级": r["taxonomy"], "来源": r["source"],
        "B2": round(composite(r["B2"]), 3),
        "B3": round(composite(r["B3"]), 3),
        "B4": round(composite(r["B4"]), 3),
    })
df_detail = pd.DataFrame(detail).set_index("id")
display(df_detail)

worst = df_detail.assign(gap=df_detail["B4"] - df_detail["B3"]).nsmallest(3, "gap")
print("B4 相对 B3 退步最多的三题（这几题最值得去看 bad case）:")
display(worst)

In [ ]:
# 落盘。评测产物要能被别人重跑核对，不能只留一张截图
out = BASE / "results"
out.mkdir(exist_ok=True)
report = {
    "provenance": PROVENANCE,
    "overall": df_overall.reset_index().to_dict("records"),
    "per_stratum": rows,
    "decision_checks": checks,
    "verdict": verdict,
    "passed": f"{passed}/{N_COMPARISONS}",
    "per_question": RESULT["per_question"],
}
(out / "h1_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False),
                                    encoding="utf-8")
print(f"已写入 {(out / 'h1_report.json').relative_to(BASE)}")

## 4.5 怎么读这个结果

我这次跑出来是这样。Part 4 是确定性的，你的机器上应该一致：

| 臂 | Recall@5 | MRR | nDCG@5 | composite |
|---|---:|---:|---:|---:|
| B2 dense-only | 0.642 | 0.617 | 0.488 | 0.582 |
| B3 hybrid + RRF | 0.737 | 0.733 | 0.587 | 0.686 |
| B4 + 多步取证 | 0.690 | 0.750 | 0.640 | 0.693 |

| 层级 | B2 | B3 | B4 | B4−B2 | B4−B3 |
|---|---:|---:|---:|---:|---:|
| L2（5 题） | 0.693 | 0.712 | 0.670 | −0.023 | −0.042 |
| L3（3 题） | 0.345 | 0.529 | 0.528 | **+0.183** | −0.000 |

**四项比较通过一项，verdict = `unsupported`。**

### 这个结果说明什么

先说它**不**说明什么：它不说明"多步取证没用"。看 L3 那行，B4 比普通向量 RAG 高 0.183，这是明确的收益。

它说明的是**收益的分布**：

- **检索本来就够强的地方（L2，B3 已经 0.712），多步取证是负收益。** 扩池会把"结构上相关、但和问题措辞不像"的代码推进前 5，挤掉本来排对了的证据。真实系统靠 cross-encoder 重排和 LLM 挑证据来吸收这个代价，这个最小复现两样都没有。
- **检索够不着的地方（L3，B3 只有 0.529），价值才显现。** 架构题要跨好几个文件，一次 top-k 根本覆盖不全。

方向上这和原项目 33 道题的结论一致：那边 L3 的 `B4−B2 = +0.247`，而 L2 上对最强 baseline 的 `B4−B3` 只有 `+0.0439`——差 0.0061 没过门槛，同样判 `unsupported`。**Agent 的价值集中在检索失效的地方，不在检索已经够用的地方。**

### 一个必须交代的过程：我试了四个配置

写这个 notebook 时，B4 的实现和 embedding 模型各有两个候选。四种组合我都跑了，harness 完全相同：

| 配置 | B2 | B3 | B4 | 四项通过 |
|---|---:|---:|---:|---|
| 通用模型 + 扩池后统一重排 | 0.508 | 0.550 | 0.521 | 0/4 |
| 通用模型 + 按来源名次排序 | 0.508 | 0.550 | **0.706** | **3/4** |
| 代码模型 + 扩池后统一重排 | 0.582 | 0.686 | 0.686 | 2/4 |
| **代码模型 + 按来源名次排序（本 notebook 采用）** | 0.582 | 0.686 | 0.693 | **1/4** |

**如果按"哪个结果好看"来选，我应该选第二行。** 它给出四项过三项——不但分数高，还正好和原项目"四项通过三项"的结论形状对上，故事更完整。

我选了最后一行。理由和 verdict 无关，只有两条工程判断：

1. **「按来源名次排序」是对的实现。** 追出来的那堆证据本来就不是一个有序列表，拿 `ast.walk` 的遍历顺序当名次是纯噪声。这一点在换模型之前就该定下来。
2. **代码检索模型才对得上原项目**（那边用 `jina-embeddings-v2-base-code`）。而且它客观上让**检索基线变强了**：B3 的 composite 从 0.550 涨到 0.686。

代价很直接：baseline 变强之后 B4 更难赢，verdict 从 3/4 掉到 1/4。

**这就是 4.3 节那条规则存在的意义。** 判据先锁死，配置按工程理由选，然后结果是什么报什么。反过来做——先看四个结果再挑一个说法——每一步都能找到听起来合理的理由，但最后得到的不是评测，是一个说服工具。

### 三条这个 notebook 复现不了、但必须知道的事

1. **确定性路径下重复运行没有信息量。** 这里三个臂都是确定性的，跑十遍结果一样。原项目的方差来自 LLM 合成环节（temperature 0.1，选哪几条引用有随机性）——那边跑三轮，决定结论的那条 margin 分别是 0.0376 / 0.0057 / 0.0884，**极差 0.083 比 0.05 的判定门槛本身还大，单轮结论会翻转**。如果只跑一轮而恰好是第三轮，就会心安理得地写下"假设成立"。

   > 如果预算只够做一件事，**跑重复比扩题更优先**。你不知道方差有多大，就不知道任何一个数字意味着什么。

2. **`B4 − B3` 混了两件事。** 它同时包含"多步取证"和"调用图"的贡献。要拆开，得再加一个**只关图工具、其余不变**的诊断臂。原项目加了 B3.5，结果是：L3 上多步取证贡献 +0.1466，调用图只有 +0.0226——**约 6.5 倍**。我在调用图上花的时间最多，孤立贡献却最小。这个结果不好受，但它是整次评测里信息量最大的数字。

   另外，11 个工具里的 6 个图工具，**有 4 个从未产出过任何被引用的证据**。工具数量不等于工具贡献。

3. **诊断臂不能反过来当判据。** B3.5 的故事比 H1 好听得多，于是会有很强的诱惑把成功标准改成 `B4 − B3.5`。不行——它是后加的诊断臂，代码里明确排除在判据之外。**消融可以改变我对系统的理解，不能改变已经锁定的判定。**

### 这套评测的边界

- 单一 Python 仓库、10 道题（原项目 33 道），单题权重很大，不能外推到其他语言和大型仓库
- 没有和通用 Coding Agent 做统一条件下的端到端对比
- 没有真实用户 A/B——"帮助新人 onboarding"目前仍然只是应用假设
- 只测"有没有找到并引用正确代码"，不测"解释是否准确、清晰、可执行"
- `graph_reverse` 的题来自系统自己的调用关系，可能漏掉它本身看不到的链路

---

# 小结

| 做了什么 | 为什么这么做 |
|---|---|
| 按 AST 语法边界切块，保留 `文件 / 符号 / 行号` | 让引用能点回源码，也让评测能自动判分 |
| BM25 + 向量双路，加权 RRF 融合 | 符号查询和语义查询覆盖面不同，单路会漏；RRF 用名次，不假设两边同分布 |
| 四个只读工具 + 多步取证 | 跨文件链路问题，一次 top-k 检索答不了 |
| 引用删除**前**计分、零引用记 0 | 堵死"多猜换高分"和"不回答得满分"两条捷径 |
| baseline 阶梯 + 预注册判据 + 消融臂 | 让"它到底好在哪"这个问题可回答，而不是靠三个成功截图 |
| 保留 `unsupported` 结论 | 评测的目的是得到一个可能推翻你的结论，不是得到一个通过 |

方法论完整版：[Extra14《垂直场景 Agent 的自建评测》](../../Extra-Chapter/Extra14-垂直场景Agent的自建评测.md)

如果你正在做毕业设计，我的建议是：**在写第一行评测代码之前，先花五分钟把成功判据写下来 commit 掉。** 这是整个项目里投入产出比最高的五分钟。